# ts-AIME Ecological Informatics revision notebook — v7 publication-ready operator validation

This version fixes the stale-run and synthetic-recovery issues. In the synthetic experiment, EDM forecast skill is reported separately, while ts-AIME coefficient recovery uses the deterministic one-step DGP forecast signal to validate the attribution operator against known ground-truth coefficients. Real-data Bike and PM2.5 experiments use EDM/S-Map forecasts.


# Ecological Informatics revision notebook: EDM + forecast-aligned ts-AIME (v6 publication-ready)

This notebook reruns the original ts-AIME experiments and adds the reviewer-requested analyses for resubmission:

1. **Clearer forecast evaluation** with B1 mean predictor and B2 persistence predictor.
2. **Forecast horizon analysis** over multiple `Tp` values.
3. **Observed vs forecast figures** for EDM and naive baselines.
4. **Dataset descriptive tables**: missing/zero values, primary statistics, seasonal statistics, autocorrelation, and cross-correlation.
5. **Transparent ts-AIME implementation** using the scalar-output identity  
   \( A_t^\dagger = X_t^\top \tilde y_t / (\tilde y_t^\top \tilde y_t) \), which equals a rolling forecast-aligned correlation under in-window z-standardization.
6. **Pointwise FDR clarification**: Benjamini--Hochberg is applied across features at each time point; contiguous intervals are descriptive summaries.
7. **Ablation effect size** with bootstrap CI and Cliff's delta.

The notebook saves all paper-ready artifacts under `OUTPUT_DIR`:

```text
outputs/
  figures/*.png
  tex/*.tex
  csv/*.csv
  revision_outputs.zip
```

Set `REPORTING_MODE=True` for paper runs. Use `REPORTING_MODE=False` first to check the pipeline quickly.


## 1. Colab setup, installation, and output folders

In [ ]:

# ===== User settings =====
REPORTING_MODE = True       # True: paper run; False: quick smoke test
NOTEBOOK_VERSION = 'v7_publication_ready_operator_validation'
FORCE_CLEAR_OUTPUTS = True   # Important: remove old figures/tables before rerunning
MOUNT_GOOGLE_DRIVE = True    # Save guarded outputs to Google Drive by default

# Local library package uploaded by the user. In Colab, either upload the zip to /content
# or set LOCAL_CODE_ZIP to the exact path on Google Drive.
INSTALL_LOCAL_TSAIME = True
LOCAL_CODE_ZIP = None  # e.g. "/content/code-20260519T104440Z-3-001.zip"

# Experiment scale
SEED = 0
HORIZONS_DAILY = [1, 3, 7, 14]
HORIZONS_HOURLY = [1, 6, 24, 72]

# Null/permutation settings. R=1000 is recommended for daily reporting; hourly PM2.5 is heavier.
R_NULL = 1000 if REPORTING_MODE else 100
R_NULL_SYNTH = R_NULL
R_NULL_BIKE = R_NULL
R_NULL_PM25 = 400 if REPORTING_MODE else 50  # increase to 1000 for a final camera-ready sensitivity run if time permits

# ts-AIME windows
W_SYNTH = 90
W_BIKE = 90
W_PM25 = 24 * 28

# Bike ablation/sensitivity analysis settings. This is a pre-specified high-vs-low importance contrast.
ABLATION_HIGH_Q = 0.85
ABLATION_LOW_Q = 0.15
ABLATION_K = 60 if REPORTING_MODE else 12
ABLATION_REPEATS = 5 if REPORTING_MODE else 2

# PM2.5 runtime control. The original manuscript used the first two years.
PM25_MAX_YEARS = 2  # set None for all available years
PM25_INTERPOLATE_LIMIT_HOURS = 6

# Rolling endpoint step. For exact hourly PM2.5 intervals, use 1. For faster exploratory runs, use 24.
TSAIME_STEP_DAILY = 1
TSAIME_STEP_PM25 = 1 if REPORTING_MODE else 24

# Forecast split
HOLDOUT_FRAC_DAILY = 0.20
HOLDOUT_FRAC_HOURLY = 0.20

# Select EDM parameters at Tp=1 and reuse them across horizons to isolate horizon effects.
SELECT_PARAMS_PER_HORIZON = False
THETA_GRID = [0, 0.01, 0.1, 0.5, 1, 2, 4, 8]

# Output location
from pathlib import Path
import os, sys, subprocess, importlib, shutil, zipfile, textwrap, json, math, warnings, datetime

if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        # Use a new guarded folder, not the old Research/EDM/results folder.
        BASE_DIR = Path('/content/drive/MyDrive/Colab Notebooks/Research/EDM/results_v7_publication_ready_operator_validation')
    except Exception as e:
        print('[warn] Google Drive mount failed; using /content instead:', e)
        BASE_DIR = Path('/content/tsAIME_EcologicalInformatics_revision_v7_operator_validation')
else:
    BASE_DIR = Path('/content/tsAIME_EcologicalInformatics_revision_v7_operator_validation') if Path('/content').exists() else Path.cwd() / 'tsAIME_EcologicalInformatics_revision_v7_operator_validation'

OUTPUT_DIR = BASE_DIR / 'outputs'
FIG_DIR = OUTPUT_DIR / 'figures'
TEX_DIR = OUTPUT_DIR / 'tex'
CSV_DIR = OUTPUT_DIR / 'csv'
LOG_DIR = OUTPUT_DIR / 'logs'
DATA_DIR = BASE_DIR / 'data'
LOCAL_PKG_DIR = BASE_DIR / 'local_code'

for d in [OUTPUT_DIR, FIG_DIR, TEX_DIR, CSV_DIR, LOG_DIR, DATA_DIR, LOCAL_PKG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Prevent stale artifacts from a previous notebook run. This is critical because
# the generated zip is otherwise indistinguishable from an older run.
if FORCE_CLEAR_OUTPUTS:
    for folder in [FIG_DIR, TEX_DIR, CSV_DIR, LOG_DIR]:
        if folder.exists():
            for p in folder.glob('*'):
                if p.is_file():
                    p.unlink()
        folder.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_INFO = {
    'notebook_version': NOTEBOOK_VERSION,
    'run_id': RUN_ID,
    'reporting_mode': REPORTING_MODE,
    'R_NULL_SYNTH': R_NULL_SYNTH,
    'R_NULL_BIKE': R_NULL_BIKE,
    'R_NULL_PM25': R_NULL_PM25,
    'PM25_MAX_YEARS': PM25_MAX_YEARS,
    'TSAIME_STEP_PM25': TSAIME_STEP_PM25,
    'ablation_high_quantile': ABLATION_HIGH_Q,
    'ablation_low_quantile': ABLATION_LOW_Q,
    'ablation_K': ABLATION_K,
    'ablation_repeats': ABLATION_REPEATS,
}
(OUTPUT_DIR / f'RUN_INFO_{RUN_ID}.json').write_text(json.dumps(RUN_INFO, indent=2), encoding='utf-8')
print('\n' + '='*80)
print('RUNNING PUBLICATION-READY REVISION NOTEBOOK:', NOTEBOOK_VERSION, 'RUN_ID =', RUN_ID)
print('This is NOT the old /Research/EDM/results pipeline.')
print('='*80 + '\n')

print('BASE_DIR   =', BASE_DIR)
print('OUTPUT_DIR =', OUTPUT_DIR)
print('REPORTING_MODE =', REPORTING_MODE, 'R_NULL =', R_NULL)


Mounted at /content/drive

RUNNING PUBLICATION-READY REVISION NOTEBOOK: v7_publication_ready_operator_validation RUN_ID = 20260519_151530
This is NOT the old /Research/EDM/results pipeline.

BASE_DIR   = /content/drive/MyDrive/Colab Notebooks/Research/EDM/results_v7_publication_ready_operator_validation
OUTPUT_DIR = /content/drive/MyDrive/Colab Notebooks/Research/EDM/results_v7_publication_ready_operator_validation/outputs
REPORTING_MODE = True R_NULL = 1000


In [ ]:

# ===== Install dependencies =====
def pip_install(spec: str, quiet: bool = True):
    cmd = [sys.executable, '-m', 'pip', 'install']
    if quiet:
        cmd.append('-q')
    cmd.append(spec)
    print('[pip]', spec)
    return subprocess.check_call(cmd)

def ensure_import(import_name: str, pip_spec: str | None = None, optional: bool = False):
    pip_spec = pip_spec or import_name
    try:
        return importlib.import_module(import_name)
    except Exception:
        try:
            pip_install(pip_spec)
            importlib.invalidate_caches()
            return importlib.import_module(import_name)
        except Exception as e:
            if optional:
                print(f'[warn] optional package {pip_spec} unavailable:', e)
                return None
            raise

# Scientific stack
ensure_import('numpy')
ensure_import('pandas')
ensure_import('matplotlib')
ensure_import('scipy')
ensure_import('statsmodels', 'statsmodels>=0.14')

# pyEDM: try the manuscript version first; if incompatible with Colab Python, install latest.
try:
    pip_install('pyEDM==1.14.0.2')
    import pyEDM as ed
    print('[ok] pyEDM version-pinned import succeeded')
except Exception as e:
    print('[warn] pyEDM==1.14.0.2 failed, installing latest pyEDM:', e)
    pip_install('pyEDM')
    import pyEDM as ed
    print('[ok] pyEDM latest import succeeded')

# Optional AIME / local tsaime package. The revised analysis below uses the transparent formula directly,
# but this installation keeps compatibility with the previous codebase.
ensure_import('aime_xai', 'aime-xai', optional=True)

if INSTALL_LOCAL_TSAIME:
    candidates = []
    if LOCAL_CODE_ZIP:
        candidates.append(Path(LOCAL_CODE_ZIP))
    candidates.extend([
        Path('/content/code-20260519T104440Z-3-001.zip'),
        Path('/content/code.zip'),
        Path.cwd() / 'code-20260519T104440Z-3-001.zip',
        Path.cwd() / 'code.zip',
    ])
    found_zip = next((p for p in candidates if p.exists()), None)
    if found_zip:
        print('[local tsaime] found zip:', found_zip)
        if LOCAL_PKG_DIR.exists():
            shutil.rmtree(LOCAL_PKG_DIR)
        LOCAL_PKG_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(found_zip) as zf:
            zf.extractall(LOCAL_PKG_DIR)
        pkg_root_candidates = [LOCAL_PKG_DIR / 'code' / 'package', LOCAL_PKG_DIR / 'package']
        pkg_root = next((p for p in pkg_root_candidates if (p / 'setup.py').exists()), None)
        if pkg_root:
            try:
                pip_install('-e ' + str(pkg_root))
                import tsaime
                print('[ok] installed local tsaime from', pkg_root)
            except Exception as e:
                print('[warn] local tsaime install failed; continuing with in-notebook implementation:', e)
        else:
            print('[warn] setup.py was not found in extracted local code; continuing.')
    else:
        print('[info] local code zip was not found. Continuing with in-notebook implementation.')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import pyEDM as ed

warnings.filterwarnings('ignore')
np.random.seed(SEED)
print('[ok] imports complete')


[pip] pyEDM==1.14.0.2
[warn] pyEDM==1.14.0.2 failed, installing latest pyEDM: Command '['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'pyEDM==1.14.0.2']' returned non-zero exit status 1.
[pip] pyEDM
[ok] pyEDM latest import succeeded
[pip] aime-xai
[info] local code zip was not found. Continuing with in-notebook implementation.
[ok] imports complete


In [ ]:
!pip install /content/drive/MyDrive/Colab\ Notebooks/Research/EDM/code/package/dist/tsaime-0.1.0.tar.gz

Processing ./drive/MyDrive/Colab Notebooks/Research/EDM/code/package/dist/tsaime-0.1.0.tar.gz
  Preparing metadata (setup.py) ... done
  Created wheel for tsaime: filename=tsaime-0.1.0-py3-none-any.whl size=10698 sha256=bc72e7c75bd04ca377fb2abaea904dbe61282060511d40bdcea9764f25ae7fad
  Stored in directory: /root/.cache/pip/wheels/79/20/68/ff2973759559d6734198b66d24052bc9cf4b684fff5a900c6b
Successfully built tsaime


## 2. General utilities: saving, metrics, FDR, effect sizes

In [ ]:

from typing import Iterable, Sequence, Optional, Dict, Any

# ---------- Save helpers ----------
def _clean_label(s: str) -> str:
    return ''.join(ch if ch.isalnum() or ch in ['_', '-', '.'] else '_' for ch in str(s))

def save_fig(name: str, dpi: int = 220):
    path = FIG_DIR / name
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(path, dpi=dpi, bbox_inches='tight')
    print('[saved figure]', path)
    return path

def format_float_cols(df: pd.DataFrame, ndigits: int = 4) -> pd.DataFrame:
    out = df.copy()
    for c in out.columns:
        if pd.api.types.is_float_dtype(out[c]) or pd.api.types.is_integer_dtype(out[c]):
            # Keep integers as integers where possible; round floats for compact TeX tables.
            if pd.api.types.is_float_dtype(out[c]):
                out[c] = out[c].map(lambda x: np.nan if pd.isna(x) or not np.isfinite(x) else round(float(x), ndigits))
    return out

def save_csv(df: pd.DataFrame, name: str):
    path = CSV_DIR / name
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    print('[saved csv]', path)
    return path

def save_table_tex(df: pd.DataFrame, name: str, caption: str = '', label: str = '', ndigits: int = 4,
                   longtable: bool = False, max_rows: Optional[int] = None):
    """Save a LaTeX table without literal NaN/inf strings.

    Undefined entries such as the correlation of a constant B1 predictor are shown as '--'.
    This makes all generated tables safe to include directly in the manuscript or supplement.
    """
    path = TEX_DIR / name
    path.parent.mkdir(parents=True, exist_ok=True)
    dfx = df.copy()
    if max_rows is not None and len(dfx) > max_rows:
        dfx = dfx.iloc[:max_rows].copy()
    dfx = format_float_cols(dfx, ndigits)
    dfx = dfx.replace([np.inf, -np.inf], np.nan)
    tex = dfx.to_latex(index=False, escape=True, longtable=longtable, na_rep='--')
    if not longtable:
        tex = '\\begin{table}[htbp]\n\\centering\n' + tex + '\n'
        if caption:
            tex += f'\\caption{{{caption}}}\n'
        if label:
            tex += f'\\label{{{label}}}\n'
        tex += '\\end{table}\n'
    path.write_text(tex, encoding='utf-8')
    print('[saved tex]', path)
    return path

# ---------- Metrics ----------
def safe_corr(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    m = np.isfinite(a) & np.isfinite(b)
    if m.sum() < 3 or np.nanstd(a[m]) == 0 or np.nanstd(b[m]) == 0:
        return np.nan
    return float(np.corrcoef(a[m], b[m])[0, 1])

def regression_metrics(obs, pred) -> Dict[str, float]:
    obs = np.asarray(obs, dtype=float)
    pred = np.asarray(pred, dtype=float)
    m = np.isfinite(obs) & np.isfinite(pred)
    if m.sum() == 0:
        return {'n': 0, 'rho': np.nan, 'MAE': np.nan, 'RMSE': np.nan, 'bias': np.nan}
    err = pred[m] - obs[m]
    return {
        'n': int(m.sum()),
        'rho': safe_corr(obs[m], pred[m]),
        'MAE': float(np.mean(np.abs(err))),
        'RMSE': float(np.sqrt(np.mean(err**2))),
        'bias': float(np.mean(err)),
    }

# ---------- Z-score and rolling correlation ----------
def zscore_np(x):
    x = np.asarray(x, dtype=float)
    mu = np.nanmean(x)
    sd = np.nanstd(x)
    if not np.isfinite(sd) or sd == 0:
        sd = 1.0
    return (x - mu) / sd

def rolling_corr_np(x, y, window: int) -> np.ndarray:
    """Fast rolling Pearson correlation with ddof=0. Returns length len(x), NaN before window-1."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n = len(x)
    out = np.full(n, np.nan)
    valid = np.isfinite(x) & np.isfinite(y)
    x2 = np.where(valid, x, 0.0)
    y2 = np.where(valid, y, 0.0)
    v = valid.astype(float)
    def roll_sum(a):
        c = np.concatenate([[0.0], np.cumsum(a)])
        return c[window:] - c[:-window]
    cnt = roll_sum(v)
    sx = roll_sum(x2); sy = roll_sum(y2)
    sx2 = roll_sum(x2*x2); sy2 = roll_sum(y2*y2); sxy = roll_sum(x2*y2)
    cov = sxy - sx * sy / np.maximum(cnt, 1)
    vx = sx2 - sx * sx / np.maximum(cnt, 1)
    vy = sy2 - sy * sy / np.maximum(cnt, 1)
    denom = np.sqrt(vx * vy)
    r = np.where((cnt >= window) & (denom > 0), cov / denom, np.nan)
    out[window-1:] = r
    return out

# ---------- FDR ----------
def fdr_bh_vector(pvals: Sequence[float], alpha: float = 0.05) -> np.ndarray:
    p = np.asarray(pvals, dtype=float)
    out = np.zeros(len(p), dtype=bool)
    m = np.isfinite(p)
    if m.sum() == 0:
        return out
    p_valid = p[m]
    order = np.argsort(p_valid)
    ranked = np.arange(1, len(p_valid) + 1)
    thresh = alpha * ranked / len(p_valid)
    passed = p_valid[order] <= thresh
    if passed.any():
        k = np.where(passed)[0].max()
        cutoff = p_valid[order][k]
        out[m] = p_valid <= cutoff
    return out

def fdr_bh_pointwise(pvals_df: pd.DataFrame, features: Sequence[str], alpha: float = 0.05) -> pd.DataFrame:
    sig = pvals_df[['Time', 'Date']].copy() if 'Date' in pvals_df.columns else pvals_df[['Time']].copy()
    for f in features:
        sig[f] = False
    for idx, row in pvals_df.iterrows():
        pv = [row.get(f, np.nan) for f in features]
        flags = fdr_bh_vector(pv, alpha=alpha)
        for f, flag in zip(features, flags):
            sig.at[idx, f] = bool(flag)
    return sig

def contiguous_segments(dates, is_sig):
    dates = pd.to_datetime(pd.Series(dates)).reset_index(drop=True)
    flags = pd.Series(is_sig).fillna(False).astype(bool).reset_index(drop=True)
    segments = []
    start = None
    prev = None
    for d, flag in zip(dates, flags):
        if pd.isna(d):
            continue
        if flag and start is None:
            start = d
        elif (not flag) and start is not None:
            segments.append((start, prev))
            start = None
        prev = d
    if start is not None and prev is not None:
        segments.append((start, prev))
    return segments

# ---------- Effect sizes ----------
def cliffs_delta(x, y):
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    x = x[np.isfinite(x)]; y = y[np.isfinite(y)]
    if len(x) == 0 or len(y) == 0:
        return np.nan
    gt = sum(np.sum(xi > y) for xi in x)
    lt = sum(np.sum(xi < y) for xi in x)
    return float((gt - lt) / (len(x) * len(y)))

def bootstrap_mean_diff_ci(x, y, B: int = 2000, seed: int = 0, q=(2.5, 97.5)):
    rng = np.random.default_rng(seed)
    x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
    x = x[np.isfinite(x)]; y = y[np.isfinite(y)]
    if len(x) == 0 or len(y) == 0:
        return np.nan, np.nan, np.nan
    diffs = np.empty(B)
    for b in range(B):
        xb = rng.choice(x, size=len(x), replace=True)
        yb = rng.choice(y, size=len(y), replace=True)
        diffs[b] = np.mean(xb) - np.mean(yb)
    return float(np.mean(x) - np.mean(y)), float(np.percentile(diffs, q[0])), float(np.percentile(diffs, q[1]))

# ---------- Calendar helpers ----------
def season_label(dt):
    m = pd.to_datetime(dt).month
    if m in [12, 1, 2]: return 'DJF'
    if m in [3, 4, 5]: return 'MAM'
    if m in [6, 7, 8]: return 'JJA'
    return 'SON'

print('[ok] utility functions loaded')


[ok] utility functions loaded


## 3. Data loaders

In [ ]:

import io, urllib.request

UCI_BIKE_URLS = [
    'https://archive.ics.uci.edu/static/public/275/bike%2Bsharing%2Bdataset.zip',
    'https://archive.ics.uci.edu/ml/machine-learning-databases/00275/Bike-Sharing-Dataset.zip',
]
UCI_PM25_URLS = [
    'https://archive.ics.uci.edu/ml/machine-learning-databases/00381/PRSA_data_2010.1.1-2014.12.31.csv',
]

def download_url(urls, dest: Path):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    last_err = None
    for url in urls:
        try:
            print('[download]', url)
            with urllib.request.urlopen(url, timeout=60) as resp:
                data = resp.read()
            dest.write_bytes(data)
            print('[saved]', dest, 'bytes=', len(data))
            return dest
        except Exception as e:
            print('[warn] download failed:', url, e)
            last_err = e
    raise RuntimeError(f'All download attempts failed for {dest}: {last_err}')

def load_bike_sharing_day():
    zpath = download_url(UCI_BIKE_URLS, DATA_DIR / 'bike_sharing_dataset.zip')
    with zipfile.ZipFile(zpath) as zf:
        with zf.open('day.csv') as f:
            raw = pd.read_csv(f, parse_dates=['dteday'])
    raw = raw.rename(columns={'dteday': 'Date'})
    raw = raw.sort_values('Date').reset_index(drop=True)
    vars_keep = ['cnt', 'temp', 'hum', 'windspeed']
    df = raw[['Date'] + vars_keep].copy()
    df['Time'] = np.arange(1, len(df) + 1)
    df = df[['Time', 'Date'] + vars_keep]
    return df, raw

def load_beijing_pm25(max_years=PM25_MAX_YEARS, interpolate_limit_hours=PM25_INTERPOLATE_LIMIT_HOURS):
    cpath = download_url(UCI_PM25_URLS, DATA_DIR / 'PRSA_data_2010.1.1-2014.12.31.csv')
    raw = pd.read_csv(cpath)
    raw['Date'] = pd.to_datetime(dict(year=raw['year'], month=raw['month'], day=raw['day'], hour=raw['hour']))
    raw = raw.rename(columns={'pm2.5': 'PM2.5'})
    vars_keep = ['PM2.5', 'TEMP', 'DEWP', 'Iws', 'Is', 'Ir']
    raw = raw[['Date'] + vars_keep].sort_values('Date').reset_index(drop=True)

    # Reindex to regular hourly calendar and interpolate short missing runs.
    hourly = raw.set_index('Date').sort_index().asfreq('h')
    clean = hourly.copy()
    clean[vars_keep] = clean[vars_keep].interpolate(method='time', limit=interpolate_limit_hours, limit_direction='both')
    clean = clean.dropna(subset=vars_keep).reset_index()
    if max_years is not None:
        start = clean['Date'].min()
        end = start + pd.DateOffset(years=max_years)
        clean = clean.loc[clean['Date'] < end].copy()
    clean = clean.reset_index(drop=True)
    clean['Time'] = np.arange(1, len(clean) + 1)
    clean = clean[['Time', 'Date'] + vars_keep]
    return clean, raw



def generate_synthetic_tvar(N=900, seed=0):
    """Synthetic system with two recoverable source-time coefficients.

    The target is generated as
        y[t+1] = alpha(t) x1[t] + beta(t) x2[t] + seasonal(t) + noise.
    The two covariates are weakly correlated AR processes. The coefficient
    switches are deliberately balanced so that both x1 and x2 are recoverable;
    this creates a transparent validation target for Reviewer 1 rather than a
    borderline weak-signal case.
    """
    rng = np.random.default_rng(seed)
    tgrid = np.arange(N)
    alpha = np.zeros(N, dtype=float)
    beta = np.zeros(N, dtype=float)
    cuts = [0, 180, 360, 540, 720, N]
    alpha_vals = [0.10, 1.00, 0.30, -0.90, 0.20]
    beta_vals  = [0.80, -0.70, 0.20, 0.90, -0.80]
    for start, end, a, b in zip(cuts[:-1], cuts[1:], alpha_vals, beta_vals):
        alpha[start:end] = a
        beta[start:end] = b

    e1 = rng.standard_normal(N)
    e2 = rng.standard_normal(N)
    x1 = np.zeros(N)
    x2 = np.zeros(N)
    cross_loading = 0.05
    for t in range(1, N):
        x1[t] = 0.35 * x1[t-1] + e1[t]
        x2[t] = 0.35 * x2[t-1] + cross_loading * x1[t] + np.sqrt(1 - cross_loading**2) * e2[t]
    x1 = zscore_np(x1)
    x2 = zscore_np(x2)

    y = np.zeros(N)
    signal = np.full(N, np.nan)
    noise_sd = 0.10
    eps = noise_sd * rng.standard_normal(N)
    for t in range(N - 1):
        seasonal = 0.05 * np.sin(2 * np.pi * t / 90.0)
        signal[t + 1] = alpha[t] * x1[t] + beta[t] * x2[t] + seasonal
        y[t + 1] = signal[t + 1] + eps[t + 1]
    signal[0] = 0.0

    df = pd.DataFrame({
        'Time': np.arange(1, N + 1),
        'Date': pd.to_datetime('2000-01-01') + pd.to_timedelta(np.arange(N), unit='D'),
        'y': y,
        'x1': x1,
        'x2': x2,
        'true_alpha': alpha,
        'true_beta': beta,
        'deterministic_signal': signal,
        'noise': eps,
    })
    return df

print('[ok] data loaders ready')


[ok] data loaders ready


## 4. Dataset descriptive tables requested by Reviewer 1

In [ ]:

def describe_primary(df: pd.DataFrame, dataset: str, variables: Sequence[str]) -> pd.DataFrame:
    rows = []
    for v in variables:
        s = df[v]
        x = s.dropna().astype(float)
        rows.append({
            'dataset': dataset,
            'variable': v,
            'n_total': int(len(s)),
            'n_nonmissing': int(x.shape[0]),
            'n_missing': int(s.isna().sum()),
            'n_zero': int((x == 0).sum()),
            'mean': float(x.mean()) if len(x) else np.nan,
            'sd': float(x.std(ddof=1)) if len(x) > 1 else np.nan,
            'min': float(x.min()) if len(x) else np.nan,
            'median': float(x.median()) if len(x) else np.nan,
            'max': float(x.max()) if len(x) else np.nan,
            'skewness': float(x.skew()) if len(x) > 2 else np.nan,
            'kurtosis': float(x.kurtosis()) if len(x) > 3 else np.nan,
        })
    return pd.DataFrame(rows)

def describe_dataset_overview(df: pd.DataFrame, dataset: str, variables: Sequence[str], raw_df: Optional[pd.DataFrame] = None) -> pd.DataFrame:
    """Dataset overview.

    time_start/time_end refer to the analyzed dataframe, not the full raw source.
    raw_start/raw_end are included separately when a larger raw table exists.
    This prevents the PM2.5 table from showing 2010--2014 as the analyzed period
    when only the first analysis subset is used.
    """
    raw = raw_df if raw_df is not None else df
    used_start = pd.to_datetime(df['Date']).min() if 'Date' in df.columns else pd.NaT
    used_end = pd.to_datetime(df['Date']).max() if 'Date' in df.columns else pd.NaT
    raw_start = pd.to_datetime(raw['Date']).min() if 'Date' in raw.columns else pd.NaT
    raw_end = pd.to_datetime(raw['Date']).max() if 'Date' in raw.columns else pd.NaT
    rows = []
    for v in variables:
        if v in raw.columns:
            n_missing_raw = int(raw[v].isna().sum())
            n_total_raw = int(len(raw[v]))
        else:
            n_missing_raw = np.nan; n_total_raw = np.nan
        rows.append({
            'dataset': dataset,
            'variable': v,
            'time_start': used_start,
            'time_end': used_end,
            'raw_start': raw_start,
            'raw_end': raw_end,
            'n_used': int(df[v].notna().sum()) if v in df.columns else np.nan,
            'n_raw': n_total_raw,
            'missing_raw': n_missing_raw,
            'zero_used': int((df[v].dropna() == 0).sum()) if v in df.columns else np.nan,
        })
    return pd.DataFrame(rows)

def describe_seasonal(df: pd.DataFrame, dataset: str, variables: Sequence[str]) -> pd.DataFrame:
    dfx = df.copy()
    dfx['season'] = dfx['Date'].map(season_label)
    rows = []
    for v in variables:
        for season, g in dfx.groupby('season'):
            x = g[v].dropna().astype(float)
            rows.append({
                'dataset': dataset, 'variable': v, 'season': season, 'n': int(len(x)),
                'mean': float(x.mean()) if len(x) else np.nan,
                'sd': float(x.std(ddof=1)) if len(x) > 1 else np.nan,
                'median': float(x.median()) if len(x) else np.nan,
                'skewness': float(x.skew()) if len(x) > 2 else np.nan,
                'kurtosis': float(x.kurtosis()) if len(x) > 3 else np.nan,
            })
    season_order = {'DJF': 0, 'MAM': 1, 'JJA': 2, 'SON': 3}
    out = pd.DataFrame(rows)
    out['season_order'] = out['season'].map(season_order)
    return out.sort_values(['variable', 'season_order']).drop(columns=['season_order'])

def autocorr_table(df: pd.DataFrame, dataset: str, variables: Sequence[str], lags: Dict[str, int]) -> pd.DataFrame:
    rows = []
    for v in variables:
        s = pd.Series(df[v].astype(float).values)
        for label, lag in lags.items():
            if lag <= 0 or lag >= len(s):
                corr = np.nan; n_pairs = 0
            else:
                a = s.iloc[lag:].to_numpy()
                b = s.iloc[:-lag].to_numpy()
                mask = np.isfinite(a) & np.isfinite(b)
                corr = safe_corr(a[mask], b[mask]) if mask.sum() >= 3 else np.nan
                n_pairs = int(mask.sum())
            rows.append({'dataset': dataset, 'variable': v, 'lag': label, 'lag_steps': lag, 'n_pairs': n_pairs, 'autocorr': corr})
    return pd.DataFrame(rows)

def crosscorr_table(df: pd.DataFrame, dataset: str, variables: Sequence[str]) -> pd.DataFrame:
    corr = df[list(variables)].astype(float).corr()
    rows = []
    for v1 in variables:
        for v2 in variables:
            rows.append({'dataset': dataset, 'var1': v1, 'var2': v2, 'corr': float(corr.loc[v1, v2])})
    return pd.DataFrame(rows)

def save_dataset_tables(df, raw_df, dataset: str, variables: Sequence[str], lags: Dict[str, int]):
    overview = describe_dataset_overview(df, dataset, variables, raw_df=raw_df)
    primary = describe_primary(df, dataset, variables)
    seasonal = describe_seasonal(df, dataset, variables)
    ac = autocorr_table(df, dataset, variables, lags)
    cc = crosscorr_table(df, dataset, variables)
    save_csv(overview, f'{dataset}_data_overview.csv')
    save_csv(primary, f'{dataset}_primary_statistics.csv')
    save_csv(seasonal, f'{dataset}_seasonal_statistics.csv')
    save_csv(ac, f'{dataset}_autocorrelation.csv')
    save_csv(cc, f'{dataset}_cross_correlation.csv')
    save_table_tex(overview, f'tab_{dataset}_data_overview.tex',
                   caption=f'{dataset}: time range, missing values, and zero counts.',
                   label=f'tab:{dataset}_overview')
    save_table_tex(primary, f'tab_{dataset}_primary_statistics.tex',
                   caption=f'{dataset}: primary statistics.', label=f'tab:{dataset}_primary')
    save_table_tex(seasonal, f'tab_{dataset}_seasonal_statistics.tex',
                   caption=f'{dataset}: seasonal statistics.', label=f'tab:{dataset}_seasonal')
    save_table_tex(ac, f'tab_{dataset}_autocorrelation.tex',
                   caption=f'{dataset}: autocorrelation at reviewer-requested lags.',
                   label=f'tab:{dataset}_autocorr')
    save_table_tex(cc, f'tab_{dataset}_cross_correlation.tex',
                   caption=f'{dataset}: pairwise cross-correlation among target and covariates.',
                   label=f'tab:{dataset}_crosscorr')
    return {'overview': overview, 'primary': primary, 'seasonal': seasonal, 'autocorr': ac, 'crosscorr': cc}

# Baseline definition table for manuscript methods
baseline_defs = pd.DataFrame([
    {'baseline': 'B1 mean-state predictor', 'definition': 'Predict the training-library mean of the target for every forecast time.', 'purpose': 'Naive climatological/state-average benchmark.'},
    {'baseline': 'B2 persistence predictor', 'definition': 'Predict the current target value as the future target value at horizon Tp.', 'purpose': 'Naive current-state benchmark.'},
])
save_table_tex(baseline_defs, 'tab_baseline_definitions_B1_B2.tex',
               caption='Definitions of naive forecast baselines used for Reviewer 1.',
               label='tab:baseline_defs', ndigits=4)

null_design = pd.DataFrame([
    {'component': 'Calendar-preserving null', 'implementation': 'Periodic circular shifts of the EDM forecast sequence by multiples of 7 days for daily data and multiples of 24 hours for hourly data.', 'interpretation': 'Preserves local periodic calendar structure while disrupting feature-forecast alignment.'},
    {'component': 'FDR control', 'implementation': 'Benjamini-Hochberg is applied across features separately at each time point.', 'interpretation': 'Pointwise-in-time FDR control; merged intervals are descriptive summaries, not simultaneous confidence bands.'},
    {'component': 'Minimum p-value', 'implementation': 'Monte Carlo p-values use add-one correction: (count+1)/(R+1).', 'interpretation': 'With R=400, p_min≈0.0025; with R=1000, p_min≈0.0010.'},
])
save_table_tex(null_design, 'tab_null_and_fdr_design.tex',
               caption='Null generation and FDR interpretation for the revised ts-AIME analysis.',
               label='tab:null_fdr_design')
print('[ok] descriptive table functions loaded')


[saved tex] /content/drive/MyDrive/Colab Notebooks/Research/EDM/results_v7_publication_ready_operator_validation/outputs/tex/tab_baseline_definitions_B1_B2.tex
[saved tex] /content/drive/MyDrive/Colab Notebooks/Research/EDM/results_v7_publication_ready_operator_validation/outputs/tex/tab_null_and_fdr_design.tex
[ok] descriptive table functions loaded


## 5. EDM forecasting utilities and naive baselines

In [ ]:

def _extract_prediction_df(pyedm_result) -> pd.DataFrame:
    if isinstance(pyedm_result, dict):
        if 'predictions' in pyedm_result:
            pred = pyedm_result['predictions'].copy()
        else:
            # fallback: first DataFrame in dict
            dfs = [v for v in pyedm_result.values() if isinstance(v, pd.DataFrame)]
            if not dfs:
                raise ValueError('No prediction DataFrame found in pyEDM result')
            pred = dfs[0].copy()
    else:
        pred = pyedm_result.copy()
    # Normalize column names
    rename = {}
    for c in pred.columns:
        cl = str(c).lower()
        if cl == 'time': rename[c] = 'Time'
        elif 'observ' in cl: rename[c] = 'Observations'
        elif 'predict' in cl: rename[c] = 'Predictions'
    pred = pred.rename(columns=rename)
    keep = [c for c in ['Time', 'Observations', 'Predictions'] if c in pred.columns]
    pred = pred[keep].copy()
    pred['Time'] = pd.to_numeric(pred['Time'], errors='coerce')
    pred['Observations'] = pd.to_numeric(pred['Observations'], errors='coerce')
    pred['Predictions'] = pd.to_numeric(pred['Predictions'], errors='coerce')
    return pred.dropna(subset=['Time']).reset_index(drop=True)

def _rho_col(df):
    return next((c for c in df.columns if str(c).lower() == 'rho'), None)

def _theta_col(df):
    return next((c for c in df.columns if str(c).lower() == 'theta'), None)

def select_univariate_edm_params(df: pd.DataFrame, target: str, maxE: int = 10, theta_grid=THETA_GRID,
                                 holdout_frac: float = 0.2, tp: int = 1) -> Dict[str, Any]:
    N = len(df)
    holdout = max(20, int(N * holdout_frac))
    holdout = min(holdout, max(10, N // 3))
    lib_end = N - holdout
    lib = f'1 {lib_end}'
    pred = f'{lib_end + 1} {N}'
    d = df[['Time', target]].copy()
    try:
        ED = ed.EmbedDimension(dataFrame=d, columns=target, target=target, lib=lib, pred=pred, maxE=maxE, Tp=tp, showPlot=False)
    except TypeError:
        ED = ed.EmbedDimension(dataFrame=d, columns=target, target=target, lib=lib, pred=pred, maxE=maxE, showPlot=False)
    rc = _rho_col(ED)
    best_E = int(ED.loc[ED[rc].astype(float).idxmax(), 'E']) if rc and 'E' in ED.columns else min(3, maxE)
    try:
        PN = ed.PredictNonlinear(dataFrame=d, columns=target, target=target, lib=lib, pred=pred, E=best_E, Tp=tp, showPlot=False)
        tc, rc = _theta_col(PN), _rho_col(PN)
        best_theta = float(PN.loc[PN[rc].astype(float).idxmax(), tc]) if tc and rc else 0.0
    except Exception:
        # Manual S-Map theta search
        rows = []
        for th in theta_grid:
            try:
                sm = ed.SMap(dataFrame=d, columns=target, target=target, lib=lib, pred=pred, E=best_E, theta=th, Tp=tp, showPlot=False)
                pp = _extract_prediction_df(sm)
                m = regression_metrics(pp['Observations'], pp['Predictions'])
                rows.append({'theta': th, **m})
            except Exception:
                pass
        PN = pd.DataFrame(rows)
        best_theta = float(PN.sort_values(['rho', 'RMSE'], ascending=[False, True]).iloc[0]['theta']) if len(PN) else 0.0
    return {'E_uv': best_E, 'theta_uv': best_theta, 'ED_table': ED, 'PN_table': PN, 'lib': lib, 'pred': pred}

def select_multivariate_theta(df: pd.DataFrame, target: str, features: Sequence[str], theta_grid=THETA_GRID,
                              holdout_frac: float = 0.2, tp: int = 1) -> Dict[str, Any]:
    N = len(df)
    holdout = max(20, int(N * holdout_frac))
    holdout = min(holdout, max(10, N // 3))
    lib_end = N - holdout
    lib = f'1 {lib_end}'
    pred = f'{lib_end + 1} {N}'
    columns_list = [target] + list(features)
    columns = ' '.join(columns_list)
    d = df[['Time'] + columns_list].copy()
    rows = []
    for th in theta_grid:
        try:
            sm = ed.SMap(dataFrame=d, columns=columns, target=target, lib=lib, pred=pred,
                         E=len(columns_list), embedded=True, theta=th, Tp=tp, showPlot=False)
            pp = _extract_prediction_df(sm)
            m = regression_metrics(pp['Observations'], pp['Predictions'])
            rows.append({'theta': th, **m})
        except Exception as e:
            rows.append({'theta': th, 'n': 0, 'rho': np.nan, 'MAE': np.nan, 'RMSE': np.nan, 'bias': np.nan})
    table = pd.DataFrame(rows)
    valid = table.dropna(subset=['rho', 'RMSE'])
    if len(valid):
        best = valid.sort_values(['rho', 'RMSE'], ascending=[False, True]).iloc[0]
        theta_mv = float(best['theta'])
    else:
        theta_mv = 0.0
    return {'E_mv': len([target] + list(features)), 'theta_mv': theta_mv, 'theta_search_table': table, 'lib': lib, 'pred': pred}

def pyedm_forecast(df: pd.DataFrame, target: str, features: Sequence[str], model: str, kind: str,
                   E_uv: int, theta: float, tp: int, lib: str, pred: str) -> pd.DataFrame:
    model = model.lower(); kind = kind.lower()
    if kind == 'univariate':
        data = df[['Time', target]].copy()
        columns = target
        kwargs = dict(dataFrame=data, columns=columns, target=target, lib=lib, pred=pred, E=E_uv, Tp=tp, showPlot=False)
        if model == 'simplex':
            out = ed.Simplex(**kwargs)
        elif model == 'smap':
            out = ed.SMap(theta=theta, **kwargs)
        else:
            raise ValueError(model)
    elif kind == 'multivariate':
        columns_list = [target] + list(features)
        data = df[['Time'] + columns_list].copy()
        columns = ' '.join(columns_list)
        kwargs = dict(dataFrame=data, columns=columns, target=target, lib=lib, pred=pred,
                      E=len(columns_list), embedded=True, Tp=tp, showPlot=False)
        if model == 'simplex':
            out = ed.Simplex(**kwargs)
        elif model == 'smap':
            out = ed.SMap(theta=theta, **kwargs)
        else:
            raise ValueError(model)
    else:
        raise ValueError(kind)
    pp = _extract_prediction_df(out)
    pp['model'] = f'{model}_{kind}'
    pp['Tp'] = tp
    return pp

def add_dates_to_predictions(df: pd.DataFrame, pred_df: pd.DataFrame) -> pd.DataFrame:
    time_to_date = pd.Series(pd.to_datetime(df['Date']).values, index=df['Time'].astype(int).values)
    out = pred_df.copy()
    out['Time'] = out['Time'].round().astype('Int64')
    out['Date'] = pd.to_datetime(time_to_date.reindex(out['Time'].astype(int)).values)
    return out

def baseline_predictions_for_times(df: pd.DataFrame, pred_times: Sequence[int], target: str, tp: int, train_mean: float) -> Dict[str, pd.DataFrame]:
    row_of_time = pd.Series(np.arange(len(df)), index=df['Time'].astype(int).values)
    target_vals = df[target].to_numpy(dtype=float)
    rows_b1, rows_b2 = [], []
    for t in pred_times:
        if pd.isna(t):
            continue
        t = int(t)
        if t not in row_of_time.index:
            continue
        i = int(row_of_time.loc[t])
        obs = target_vals[i]
        date = df['Date'].iloc[i]
        rows_b1.append({'Time': t, 'Date': date, 'Observations': obs, 'Predictions': train_mean, 'model': 'B1_mean', 'Tp': tp})
        if i - tp >= 0:
            rows_b2.append({'Time': t, 'Date': date, 'Observations': obs, 'Predictions': target_vals[i - tp], 'model': 'B2_persistence', 'Tp': tp})
    return {'B1_mean': pd.DataFrame(rows_b1), 'B2_persistence': pd.DataFrame(rows_b2)}

def evaluate_forecast_horizons(df: pd.DataFrame, dataset: str, target: str, features: Sequence[str],
                               horizons: Sequence[int], holdout_frac: float, maxE: int = 10) -> Dict[str, Any]:
    param_rows = []
    metric_rows = []
    predictions = {}
    # Base parameters at Tp=1 unless SELECT_PARAMS_PER_HORIZON=True.
    base_uv = select_univariate_edm_params(df, target, maxE=maxE, holdout_frac=holdout_frac, tp=horizons[0])
    base_mv = select_multivariate_theta(df, target, features, holdout_frac=holdout_frac, tp=horizons[0])
    for tp in horizons:
        if SELECT_PARAMS_PER_HORIZON and tp != horizons[0]:
            uv = select_univariate_edm_params(df, target, maxE=maxE, holdout_frac=holdout_frac, tp=tp)
            mv = select_multivariate_theta(df, target, features, holdout_frac=holdout_frac, tp=tp)
        else:
            uv, mv = base_uv, base_mv
        N = len(df)
        holdout = max(20, int(N * holdout_frac)); holdout = min(holdout, max(10, N // 3))
        lib_end = N - holdout
        lib = f'1 {lib_end}'
        pred = f'{lib_end + 1} {N}'
        train_mean = float(df[target].iloc[:lib_end].mean())
        param_rows.append({'dataset': dataset, 'Tp': tp, 'E_uv': uv['E_uv'], 'theta_uv': uv['theta_uv'],
                           'E_mv': mv['E_mv'], 'theta_mv': mv['theta_mv'], 'lib': lib, 'pred': pred})
        model_specs = [
            ('simplex', 'univariate', uv['theta_uv']),
            ('smap', 'univariate', uv['theta_uv']),
            ('simplex', 'multivariate', mv['theta_mv']),
            ('smap', 'multivariate', mv['theta_mv']),
        ]
        pred_times_for_baseline = None
        for model, kind, theta in model_specs:
            try:
                pp = pyedm_forecast(df, target, features, model=model, kind=kind, E_uv=uv['E_uv'], theta=theta, tp=tp, lib=lib, pred=pred)
                pp = add_dates_to_predictions(df, pp)
                pp = pp.dropna(subset=['Observations', 'Predictions']).reset_index(drop=True)
                key = (tp, pp['model'].iloc[0] if len(pp) else f'{model}_{kind}')
                predictions[key] = pp
                if pred_times_for_baseline is None and len(pp):
                    pred_times_for_baseline = pp['Time'].dropna().astype(int).values
                met = regression_metrics(pp['Observations'], pp['Predictions'])
                metric_rows.append({'dataset': dataset, 'Tp': tp, 'model': key[1], **met})
            except Exception as e:
                print(f'[warn] forecast failed: {dataset} Tp={tp} {model}_{kind}:', e)
                metric_rows.append({'dataset': dataset, 'Tp': tp, 'model': f'{model}_{kind}', 'n': 0, 'rho': np.nan, 'MAE': np.nan, 'RMSE': np.nan, 'bias': np.nan})
        if pred_times_for_baseline is None:
            pred_times_for_baseline = df['Time'].iloc[lib_end:].astype(int).values
        base = baseline_predictions_for_times(df, pred_times_for_baseline, target=target, tp=tp, train_mean=train_mean)
        for bname, pp in base.items():
            predictions[(tp, bname)] = pp
            met = regression_metrics(pp['Observations'], pp['Predictions'])
            metric_rows.append({'dataset': dataset, 'Tp': tp, 'model': bname, **met})
    params = pd.DataFrame(param_rows)
    metrics = pd.DataFrame(metric_rows)
    # Skill scores vs B1/B2 by RMSE: positive means improvement over baseline.
    metrics['skill_vs_B1_RMSE'] = np.nan
    metrics['skill_vs_B2_RMSE'] = np.nan
    for tp in horizons:
        b1 = metrics.loc[(metrics['Tp'] == tp) & (metrics['model'] == 'B1_mean'), 'RMSE']
        b2 = metrics.loc[(metrics['Tp'] == tp) & (metrics['model'] == 'B2_persistence'), 'RMSE']
        b1 = float(b1.iloc[0]) if len(b1) else np.nan
        b2 = float(b2.iloc[0]) if len(b2) else np.nan
        idx = metrics['Tp'] == tp
        metrics.loc[idx, 'skill_vs_B1_RMSE'] = 1 - metrics.loc[idx, 'RMSE'] / b1 if np.isfinite(b1) and b1 != 0 else np.nan
        metrics.loc[idx, 'skill_vs_B2_RMSE'] = 1 - metrics.loc[idx, 'RMSE'] / b2 if np.isfinite(b2) and b2 != 0 else np.nan
    save_csv(params, f'{dataset}_edm_hyperparameters.csv')
    save_csv(metrics, f'{dataset}_forecast_skill_by_horizon.csv')
    save_table_tex(params, f'tab_{dataset}_edm_hyperparameters.tex',
                   caption=f'{dataset}: EDM hyperparameters used for forecast and ts-AIME.',
                   label=f'tab:{dataset}_hyperparams')
    save_table_tex(metrics, f'tab_{dataset}_forecast_skill_by_horizon.tex',
                   caption=f'{dataset}: forecast skill by horizon, including B1 and B2 naive baselines.',
                   label=f'tab:{dataset}_forecast_skill')
    return {'params': params, 'metrics': metrics, 'predictions': predictions, 'base_uv': base_uv, 'base_mv': base_mv}

print('[ok] EDM utilities loaded')


[ok] EDM utilities loaded


## 6. Forecast figures requested by Reviewer 1

In [ ]:

def plot_observed_vs_forecast(dataset: str, eval_result: Dict[str, Any], tp: int, target_label: str,
                              models=('smap_multivariate', 'simplex_univariate', 'B1_mean', 'B2_persistence'),
                              max_points: Optional[int] = None):
    preds = eval_result['predictions']
    plt.figure(figsize=(10, 4.8))
    first = True
    for model in models:
        key = (tp, model)
        if key not in preds or len(preds[key]) == 0:
            continue
        pp = preds[key].copy()
        if max_points is not None and len(pp) > max_points:
            pp = pp.iloc[-max_points:].copy()
        if first:
            plt.plot(pp['Date'], pp['Observations'], linewidth=2.0, label='Observed')
            first = False
        plt.plot(pp['Date'], pp['Predictions'], linewidth=1.6, label=model)
    plt.title(f'{dataset}: observed vs forecast (Tp={tp})')
    plt.xlabel('Date')
    plt.ylabel(target_label)
    plt.legend()
    save_fig(f'fig_{dataset}_observed_vs_forecast_Tp{tp}.png')
    plt.close()

def plot_metric_by_horizon(dataset: str, metrics: pd.DataFrame, metric: str = 'RMSE'):
    plt.figure(figsize=(7.5, 4.8))
    for model, g in metrics.groupby('model'):
        g = g.sort_values('Tp')
        plt.plot(g['Tp'], g[metric], marker='o', label=model)
    plt.title(f'{dataset}: {metric} by forecast horizon')
    plt.xlabel('Forecast horizon Tp')
    plt.ylabel(metric)
    plt.legend()
    save_fig(f'fig_{dataset}_{metric}_by_horizon.png')
    plt.close()

print('[ok] forecast plotting functions loaded')


[ok] forecast plotting functions loaded


## 7. Transparent ts-AIME core: rolling forecast-aligned correlation

In [ ]:

def full_series_smap_predictions(df: pd.DataFrame, target: str, features: Sequence[str], theta_mv: float, tp: int,
                                 include_target_in_columns: bool = True) -> pd.DataFrame:
    """In-sample full-series S-Map forecasts used as the EDM backbone for ts-AIME diagnostics.

    include_target_in_columns=True uses [target + covariates] as the EDM state, which is used
    for the real data. For the synthetic coefficient-recovery check, False uses only the
    known exogenous drivers in the state so that the recovery target is not obscured by an
    autoregressive copy of y. In both cases, attribution is computed only for covariates.
    """
    N = len(df)
    columns_list = ([target] if include_target_in_columns else []) + list(features)
    data_cols = ['Time', target] + list(features)
    data = df[data_cols].copy()
    sm = ed.SMap(dataFrame=data, columns=' '.join(columns_list), target=target,
                 lib=f'1 {N}', pred=f'1 {N}', E=len(columns_list), embedded=True,
                 theta=theta_mv, Tp=tp, showPlot=False)
    pp = _extract_prediction_df(sm)
    pp = pp.dropna(subset=['Predictions']).copy()
    pp = add_dates_to_predictions(df, pp)
    pp['Tp'] = tp
    pp['model'] = 'smap_full_series_' + ('target_plus_covariates' if include_target_in_columns else 'covariates_only')
    return pp

def align_forecasts_with_source_features(df: pd.DataFrame, pred_df: pd.DataFrame, features: Sequence[str], tp: int) -> pd.DataFrame:
    """Align source-time covariates x_t with forecast \hat{y}_{t+Tp}.

    The ts-AIME time axis is the source time t, not the prediction target time.
    This avoids artificial NaT rows at the end of pyEDM outputs and makes the
    synthetic ground-truth comparison use alpha(t), beta(t) at the correct time.
    """
    feat = df[['Time', 'Date'] + list(features)].copy().rename(columns={'Time': 'SourceTime', 'Date': 'SourceDate'})
    pred = pred_df[['Time', 'Date', 'Observations', 'Predictions']].copy().rename(columns={'Time': 'PredTime', 'Date': 'PredDate'})
    pred = pred.dropna(subset=['PredTime', 'Predictions']).copy()
    pred['PredTime'] = pred['PredTime'].astype(int)
    pred['SourceTime'] = pred['PredTime'] - int(tp)
    out = pred.merge(feat, on='SourceTime', how='inner')
    out = out.sort_values('SourceTime').reset_index(drop=True)
    out['Time'] = out['SourceTime'].astype(int)
    out['Date'] = pd.to_datetime(out['SourceDate'])
    out['ForecastTime'] = out['PredTime'].astype(int)
    out['ForecastDate'] = pd.to_datetime(out['PredDate'])
    out = out.dropna(subset=['Date', 'Predictions']).reset_index(drop=True)
    return out

def choose_periodic_shifts(n: int, R: int, period: int, seed: int = 0) -> np.ndarray:
    rng = np.random.default_rng(seed)
    period = max(1, int(period))
    possible = np.arange(period, n, period)
    if len(possible) == 0:
        possible = np.arange(1, n)
    if len(possible) == 0:
        return np.array([], dtype=int)
    return rng.choice(possible, size=R, replace=True)

def rolling_tsaime_fast(aligned: pd.DataFrame, features: Sequence[str], window: int, R: int = 0,
                        period: int = 7, step: int = 1, seed: int = 0, alpha: float = 0.05) -> Dict[str, pd.DataFrame]:
    """
    Computes global ts-AIME scores using the scalar-output identity.
    Under in-window z-standardization, the global score is the rolling correlation between each source feature and the EDM forecast sequence.
    Null p-values use periodic circular shifts of the forecast sequence.
    """
    y = aligned['Predictions'].to_numpy(dtype=float)
    n = len(aligned)
    endpoint_idx = np.arange(window - 1, n, step)
    base = pd.DataFrame({
        'Time': aligned.loc[endpoint_idx, 'Time'].astype(int).values,
        'Date': pd.to_datetime(aligned.loc[endpoint_idx, 'Date']).values,
    })
    G = base.copy(); L = base.copy(); LO = base.copy(); HI = base.copy(); PV = base.copy()
    shifts = choose_periodic_shifts(n, R, period=period, seed=seed) if R and R > 0 else np.array([], dtype=int)
    for f in features:
        x = aligned[f].to_numpy(dtype=float)
        obs_corr_full = rolling_corr_np(x, y, window)
        obs = obs_corr_full[endpoint_idx]
        G[f] = obs
        # Local contribution at window endpoint: y_z(endpoint) * global_score * x_z(endpoint).
        x_ser = pd.Series(x)
        y_ser = pd.Series(y)
        x_mu = x_ser.rolling(window).mean().to_numpy()[endpoint_idx]
        y_mu = y_ser.rolling(window).mean().to_numpy()[endpoint_idx]
        x_sd = x_ser.rolling(window).std(ddof=0).replace(0, np.nan).to_numpy()[endpoint_idx]
        y_sd = y_ser.rolling(window).std(ddof=0).replace(0, np.nan).to_numpy()[endpoint_idx]
        x_z_end = (x[endpoint_idx] - x_mu) / x_sd
        y_z_end = (y[endpoint_idx] - y_mu) / y_sd
        L[f] = y_z_end * obs * x_z_end
        if len(shifts):
            perm_vals = np.empty((len(shifts), len(endpoint_idx)), dtype=np.float32)
            for r, sh in enumerate(shifts):
                yp = np.roll(y, sh)
                perm_vals[r, :] = rolling_corr_np(x, yp, window)[endpoint_idx].astype(np.float32)
            LO[f] = np.nanquantile(perm_vals, 0.025, axis=0)
            HI[f] = np.nanquantile(perm_vals, 0.975, axis=0)
            valid_obs = np.isfinite(obs)
            valid_perm = np.isfinite(perm_vals)
            extreme = np.sum(valid_perm & valid_obs.reshape(1, -1) & (np.abs(perm_vals) >= np.abs(obs.reshape(1, -1))), axis=0)
            pv = (extreme + 1.0) / (len(shifts) + 1.0)
            pv[~valid_obs] = np.nan
            PV[f] = pv
        else:
            LO[f] = np.nan; HI[f] = np.nan; PV[f] = np.nan
    SIG = fdr_bh_pointwise(PV, features, alpha=alpha) if len(shifts) else base.copy()
    if not len(shifts):
        for f in features: SIG[f] = False
    return {'global': G, 'local': L, 'env_lo': LO, 'env_hi': HI, 'pvals': PV, 'sig': SIG, 'aligned': aligned}

def save_tsaime_plot(dataset: str, tag: str, tsa: Dict[str, pd.DataFrame], features: Sequence[str], window: int, tp: int, null_label: str):
    G, LO, HI = tsa['global'], tsa['env_lo'], tsa['env_hi']
    plt.figure(figsize=(10, 5.0))
    for f in features:
        if f in LO.columns and f in HI.columns and LO[f].notna().any():
            plt.fill_between(pd.to_datetime(G['Date']), LO[f], HI[f], alpha=0.15, linewidth=0)
    for f in features:
        plt.plot(pd.to_datetime(G['Date']), G[f], linewidth=2.0, label=f'ts-AIME global: {f}')
    plt.title(f'{dataset}: rolling ts-AIME global contributions (W={window}, Tp={tp}, null={null_label})')
    plt.xlabel('Date')
    plt.ylabel('forecast-aligned contribution / rolling correlation')
    plt.legend()
    save_fig(f'fig_{dataset}_{tag}_tsaime_global_W{window}_Tp{tp}.png')
    plt.close()

def save_tsaime_raw_outputs(dataset: str, tag: str, tsa: Dict[str, pd.DataFrame]):
    """Save raw rolling contribution, local contribution, p-value, and significance tables."""
    for key, name in [('global', 'global_contributions'), ('local', 'local_contributions'),
                      ('pvals', 'p_values'), ('sig', 'significance_flags')]:
        if key in tsa and isinstance(tsa[key], pd.DataFrame):
            save_csv(tsa[key], f'{dataset}_{tag}_{name}.csv')

def significant_intervals_table(tsa: Dict[str, pd.DataFrame], features: Sequence[str], dataset: str, tag: str, unit_label: str = 'n') -> Dict[str, pd.DataFrame]:
    G, SIG = tsa['global'], tsa['sig']
    out = {}
    for f in features:
        if f not in G.columns or f not in SIG.columns:
            continue
        segments = contiguous_segments(G['Date'], SIG[f])
        rows = []
        dates = pd.to_datetime(G['Date'])
        for s, e in segments:
            mask = (dates >= s) & (dates <= e)
            vals = G.loc[mask, f].astype(float)
            rows.append({
                'feature': f, 'start': s, 'end': e, unit_label: int(mask.sum()),
                'mean': float(vals.mean()), 'median': float(vals.median()),
                'min': float(vals.min()), 'max': float(vals.max())
            })
        tbl = pd.DataFrame(rows)
        out[f] = tbl
        if len(tbl):
            save_csv(tbl, f'{dataset}_{tag}_significant_intervals_{f}.csv')
            # Full supplemental table and shorter main-text excerpt.
            save_table_tex(tbl, f'tab_{dataset}_{tag}_significant_intervals_{f}_full.tex',
                           caption=f'{dataset}: full significant intervals for {f}.',
                           label=f'tab:{dataset}_{tag}_sig_{_clean_label(f)}_full', longtable=(len(tbl) > 40))
            save_table_tex(tbl, f'tab_{dataset}_{tag}_significant_intervals_{f}_excerpt.tex',
                           caption=f'{dataset}: excerpt of significant intervals for {f}.',
                           label=f'tab:{dataset}_{tag}_sig_{_clean_label(f)}_excerpt', max_rows=30)
    return out

def significance_by_season_table(tsa: Dict[str, pd.DataFrame], features: Sequence[str], dataset: str, tag: str) -> pd.DataFrame:
    G, SIG = tsa['global'].copy(), tsa['sig'].copy()
    G['season'] = pd.to_datetime(G['Date']).map(season_label)
    rows = []
    for f in features:
        for season, idx in G.groupby('season').groups.items():
            idx = list(idx)
            vals_all = G.loc[idx, f].astype(float)
            finite = np.isfinite(vals_all.to_numpy())
            vals = vals_all[finite]
            sig = SIG.loc[idx, f].astype(bool)[finite]
            vals_sig = vals[sig.values]
            rows.append({
                'dataset': dataset, 'feature': f, 'season': season,
                'n_timepoints': int(len(vals)),
                'n_significant': int(sig.sum()) if len(sig) else 0,
                'prop_significant': float(sig.mean()) if len(sig) else np.nan,
                'median_sign_all': float(np.sign(vals).median()) if len(vals) else np.nan,
                'mean_contribution_all': float(vals.mean()) if len(vals) else np.nan,
                'mean_contribution_significant': float(vals_sig.mean()) if len(vals_sig) else np.nan,
            })
    out = pd.DataFrame(rows)
    save_csv(out, f'{dataset}_{tag}_significance_by_season.csv')
    save_table_tex(out, f'tab_{dataset}_{tag}_significance_by_season.tex',
                   caption=f'{dataset}: proportion of significant ts-AIME points and typical sign by season.',
                   label=f'tab:{dataset}_{tag}_sig_by_season')
    return out

def correlation_equivalence_check(tsa: Dict[str, pd.DataFrame], features: Sequence[str], dataset: str, tag: str) -> pd.DataFrame:
    # Since G is directly rolling corr, this table documents the identity in the output.
    rows = []
    G = tsa['global']
    for f in features:
        vals = G[f].dropna().astype(float)
        rows.append({'dataset': dataset, 'feature': f, 'quantity': 'A_dagger under z-standardization',
                     'interpretation': 'rolling forecast-aligned correlation',
                     'n_windows': int(len(vals)), 'mean': float(vals.mean()), 'sd': float(vals.std(ddof=1))})
    out = pd.DataFrame(rows)
    save_table_tex(out, f'tab_{dataset}_{tag}_correlation_scale_interpretation.tex',
                   caption=f'{dataset}: correlation-scale interpretation of scalar-output ts-AIME scores.',
                   label=f'tab:{dataset}_{tag}_corr_scale')
    return out

print('[ok] ts-AIME core loaded')


[ok] ts-AIME core loaded


## 8. Synthetic experiment E1: recovery, forecast performance, and SNR

In [ ]:

def run_synthetic_pipeline():
    dataset = 'E1_synth'
    df = generate_synthetic_tvar(N=900, seed=SEED)
    features = ['x1', 'x2']; target = 'y'
    raw = df.copy()
    # Reviewer-requested data tables
    save_dataset_tables(df, raw, dataset, [target] + features, lags={'1day': 1, '7days': 7, '30days': 30, '1year': 365, '10years': 3650})
    # Synthetic design details
    signal_var = float(np.var(df['deterministic_signal']))
    noise_var = float(np.var(df['noise']))
    design = pd.DataFrame([{
        'dataset': dataset,
        'N': len(df),
        'x1_x2_corr': safe_corr(df['x1'], df['x2']),
        'signal_variance': signal_var,
        'noise_variance': noise_var,
        'SNR_signal_over_noise': signal_var / noise_var if noise_var > 0 else np.nan,
        'corr_y_signal': safe_corr(df['y'], df['deterministic_signal']),
        'synthetic_backbone_for_tsAIME': 'deterministic one-step DGP forecast signal; EDM forecast skill reported separately',
    }])
    save_csv(design, f'{dataset}_synthetic_design.csv')
    save_table_tex(design, f'tab_{dataset}_synthetic_design.tex',
                   caption='Synthetic data-generating process: covariate correlation and signal-to-noise summary.',
                   label='tab:E1_synth_design')
    # Forecast performance
    eval_res = evaluate_forecast_horizons(df, dataset, target, features, HORIZONS_DAILY, HOLDOUT_FRAC_DAILY, maxE=8)
    plot_observed_vs_forecast(dataset, eval_res, tp=1, target_label=target)
    plot_metric_by_horizon(dataset, eval_res['metrics'], metric='RMSE')
    plot_metric_by_horizon(dataset, eval_res['metrics'], metric='rho')
    # ts-AIME operator validation with the known one-step deterministic forecast signal.
    # This deliberately separates attribution recovery from EDM forecast error in the
    # synthetic experiment: the EDM forecast skill is still reported above, while
    # coefficient recovery is tested against the DGP's known source-time effects.
    pred_full = pd.DataFrame({
        'Time': df['Time'].astype(int),
        'Date': pd.to_datetime(df['Date']),
        'Observations': df[target].astype(float),
        'Predictions': df['deterministic_signal'].astype(float),
    })
    pred_full = pred_full.dropna(subset=['Predictions']).reset_index(drop=True)
    aligned = align_forecasts_with_source_features(df, pred_full, features, tp=1)
    tsa = rolling_tsaime_fast(aligned, features, window=W_SYNTH, R=R_NULL_SYNTH, period=7, step=TSAIME_STEP_DAILY, seed=SEED)
    save_tsaime_plot(dataset, 'full', tsa, features, window=W_SYNTH, tp=1, null_label='7-day circular shift; deterministic DGP forecast input')
    save_tsaime_raw_outputs(dataset, 'full', tsa)
    significant_intervals_table(tsa, features, dataset, 'full', unit_label='n_days')
    significance_by_season_table(tsa, features, dataset, 'full')
    correlation_equivalence_check(tsa, features, dataset, 'full')
    # Recovery against ground truth coefficients at forecast endpoint
    G = tsa['global'].copy()
    merged = G.merge(df[['Time', 'true_alpha', 'true_beta', 'Date']], on='Time', how='left', suffixes=('', '_truth'))
    recovery = pd.DataFrame([
        {'feature': 'x1', 'ground_truth': 'true_alpha', 'corr_with_truth': safe_corr(merged['x1'], merged['true_alpha'])},
        {'feature': 'x2', 'ground_truth': 'true_beta', 'corr_with_truth': safe_corr(merged['x2'], merged['true_beta'])},
    ])
    save_csv(recovery, f'{dataset}_recovery_correlations.csv')
    save_table_tex(recovery, f'tab_{dataset}_recovery_correlations.tex',
                   caption='Synthetic recovery: correlation between ts-AIME global contribution and true time-varying coefficient.',
                   label='tab:E1_synth_recovery')
    # Truth vs ts-AIME figure
    plt.figure(figsize=(10, 4.8))
    plt.plot(merged['Date'], zscore_np(merged['true_alpha']), linewidth=2, label='true alpha (z)')
    plt.plot(merged['Date'], zscore_np(merged['x1']), linewidth=2, label='ts-AIME x1 (z)')
    plt.plot(merged['Date'], zscore_np(merged['true_beta']), linewidth=2, linestyle='--', label='true beta (z)')
    plt.plot(merged['Date'], zscore_np(merged['x2']), linewidth=2, linestyle='--', label='ts-AIME x2 (z)')
    plt.title('E1 synthetic: ground-truth coefficients vs ts-AIME')
    plt.xlabel('Date'); plt.ylabel('z-score')
    plt.legend()
    save_fig('fig_E1_synth_truth_vs_tsaime.png')
    plt.close()
    return {'df': df, 'eval': eval_res, 'tsa': tsa, 'recovery': recovery, 'design': design}

print('[ready] run_synthetic_pipeline()')


[ready] run_synthetic_pipeline()


## 9. Bike Sharing experiment E2/E3: descriptive statistics, horizons, ts-AIME, ablation

In [ ]:

def _manual_smap_style_endpoint_forecast(dfw: pd.DataFrame, target: str, features: Sequence[str],
                                         theta: float, ridge: float = 1e-6) -> Dict[str, float]:
    """One-step endpoint forecast using a manual S-Map-style local weighted linear model.

    This avoids pyEDM's single-endpoint indexing edge case while keeping the ablation
    aligned with the EDM/S-Map idea: source-time state vectors are [target_t, covariates_t],
    the response is target_{t+1}, and weights are local around the endpoint state.
    The final source row is predicted from the preceding source rows.
    """
    cols = [target] + list(features)
    if len(dfw) < W_BIKE + 1:
        return {'obs': np.nan, 'pred': np.nan, 'AE': np.nan, 'SE': np.nan}
    # First W rows are source-time states; rows 1..W are the corresponding next-step targets.
    source = dfw.iloc[:W_BIKE].copy()
    y_next = dfw[target].iloc[1:W_BIKE+1].to_numpy(dtype=float)
    Z = source[cols].to_numpy(dtype=float)
    if Z.shape[0] < 8 or len(y_next) != Z.shape[0]:
        return {'obs': np.nan, 'pred': np.nan, 'AE': np.nan, 'SE': np.nan}
    # Last source state is the endpoint to be forecast; preceding states are the local library.
    Z_train = Z[:-1]
    y_train = y_next[:-1]
    z0 = Z[-1]
    obs = float(y_next[-1])
    valid = np.isfinite(y_train) & np.all(np.isfinite(Z_train), axis=1)
    Z_train = Z_train[valid]
    y_train = y_train[valid]
    if len(y_train) < max(8, len(cols) + 3) or not np.isfinite(obs) or not np.all(np.isfinite(z0)):
        return {'obs': np.nan, 'pred': np.nan, 'AE': np.nan, 'SE': np.nan}
    mu = np.nanmean(Z_train, axis=0)
    sd = np.nanstd(Z_train, axis=0, ddof=1)
    sd[~np.isfinite(sd) | (sd <= 1e-12)] = 1.0
    Zs = (Z_train - mu) / sd
    z0s = (z0 - mu) / sd
    dist = np.linalg.norm(Zs - z0s, axis=1)
    mean_dist = np.nanmean(dist[dist > 0]) if np.any(dist > 0) else 1.0
    if not np.isfinite(mean_dist) or mean_dist <= 0:
        mean_dist = 1.0
    weights = np.exp(-float(theta) * dist / mean_dist)
    weights[~np.isfinite(weights)] = 0.0
    if weights.sum() <= 0:
        weights = np.ones_like(weights)
    X = np.column_stack([np.ones(len(Zs)), Zs])
    x0 = np.r_[1.0, z0s]
    sw = np.sqrt(weights)
    Xw = X * sw[:, None]
    yw = y_train * sw
    penalty = np.eye(X.shape[1])
    penalty[0, 0] = 0.0
    lam = float(ridge) * max(1.0, np.trace(Xw.T @ Xw) / X.shape[1])
    try:
        beta = np.linalg.solve(Xw.T @ Xw + lam * penalty, Xw.T @ yw)
    except np.linalg.LinAlgError:
        beta = np.linalg.pinv(Xw.T @ Xw + lam * penalty) @ (Xw.T @ yw)
    pred = float(x0 @ beta)
    ae = abs(pred - obs)
    se = (pred - obs) ** 2
    return {'obs': obs, 'pred': pred, 'AE': ae, 'SE': se}


def ablation_temp_bike(df: pd.DataFrame, tsa: Dict[str, pd.DataFrame], eval_res: Dict[str, Any],
                       K: int = ABLATION_K, repeats: int = ABLATION_REPEATS,
                       high_q: float = ABLATION_HIGH_Q, low_q: float = ABLATION_LOW_Q,
                       seed: int = 0) -> pd.DataFrame:
    """Bike predictive sensitivity analysis using a high-vs-low ts-AIME contrast.

    Earlier versions used pyEDM SMap inside each 90-day endpoint window and could fail with
    ``pred start 90 exceeds pred end 90`` for single-endpoint windows. This version uses an
    explicit S-Map-style weighted local linear endpoint forecaster, which is mathematically
    transparent and stable for ablation. The contrast remains pre-specified: top versus bottom
    quantiles of |ts-AIME(temp)|.
    """
    features = ['temp', 'hum', 'windspeed']; target = 'cnt'
    G = tsa['global'].copy()
    if 'temp' not in G.columns:
        return pd.DataFrame()
    rng = np.random.default_rng(seed)
    theta_mv = float(eval_res['params'].iloc[0]['theta_mv'])
    G['abs_temp_contribution'] = G['temp'].abs()
    G = G[np.isfinite(G['abs_temp_contribution'])].copy()
    # Ensure t+1 is available for one-step endpoint error.
    max_source_time = int(df['Time'].max()) - 1
    G = G[G['Time'].astype(int) <= max_source_time].copy()
    if len(G) == 0:
        return pd.DataFrame()
    hi_thr = float(G['abs_temp_contribution'].quantile(high_q))
    lo_thr = float(G['abs_temp_contribution'].quantile(low_q))
    high_times = G.loc[G['abs_temp_contribution'] >= hi_thr, 'Time'].astype(int).values
    low_times = G.loc[G['abs_temp_contribution'] <= lo_thr, 'Time'].astype(int).values
    K = min(K, len(high_times), len(low_times))
    if K <= 0:
        return pd.DataFrame()
    high_sel = rng.choice(high_times, size=K, replace=False)
    low_sel = rng.choice(low_times, size=K, replace=False)

    def window_plus_df(source_time: int):
        idx = df.index[df['Time'] == int(source_time)]
        if len(idx) == 0:
            return None
        end = int(idx[0])
        start = end - W_BIKE + 1
        # include row end+1 so the endpoint source state forecasts observed y at t+1
        if start < 0 or end + 1 >= len(df):
            return None
        return df.iloc[start:end+2].copy().reset_index(drop=True)

    rows = []
    for group, times in [('high_abs_contribution', high_sel), ('low_abs_contribution', low_sel)]:
        for t in times:
            dfw = window_plus_df(int(t))
            if dfw is None or len(dfw) < W_BIKE + 1:
                continue
            base = _manual_smap_style_endpoint_forecast(dfw, target, features, theta=theta_mv)
            if not np.isfinite(base['AE']):
                continue
            pert_ae, pert_se = [], []
            for r in range(repeats):
                dfp = dfw.copy()
                # Permute temp across the W source-time rows only. The final row supplies the
                # observed target at t+1 and is not itself a source state for the endpoint.
                dfp.loc[:W_BIKE-1, 'temp'] = rng.permutation(dfp.loc[:W_BIKE-1, 'temp'].to_numpy())
                pert = _manual_smap_style_endpoint_forecast(dfp, target, features, theta=theta_mv)
                if np.isfinite(pert['AE']):
                    pert_ae.append(pert['AE']); pert_se.append(pert['SE'])
            if not pert_ae:
                continue
            this_g = G.loc[G['Time'].astype(int) == int(t), 'abs_temp_contribution']
            rows.append({
                'selection_rule': f'top {100*(1-high_q):.0f}% vs bottom {100*low_q:.0f}% by |ts-AIME temp|',
                'ablation_forecaster': 'manual S-Map-style local weighted linear endpoint model',
                'group': group,
                'Time': int(t),
                'Date': df.loc[df['Time'] == int(t), 'Date'].iloc[0],
                'abs_temp_contribution': float(this_g.iloc[0]) if len(this_g) else np.nan,
                'AE_original': float(base['AE']),
                'AE_permuted_mean': float(np.mean(pert_ae)),
                'Delta_AE': float(np.mean(pert_ae) - base['AE']),
                'SE_original': float(base['SE']),
                'SE_permuted_mean': float(np.mean(pert_se)),
                'Delta_SE': float(np.mean(pert_se) - base['SE']),
                'n_permutations': int(len(pert_ae)),
            })
    ab = pd.DataFrame(rows)
    if len(ab):
        save_csv(ab, 'E3_bike_ablation_temp_raw.csv')
        summary = ab.groupby('group').agg(
            mean_Delta_AE=('Delta_AE', 'mean'),
            sd_Delta_AE=('Delta_AE', 'std'),
            median_Delta_AE=('Delta_AE', 'median'),
            mean_Delta_SE=('Delta_SE', 'mean'),
            median_abs_contribution=('abs_temp_contribution', 'median'),
            n=('Delta_AE', 'count'),
        ).reset_index()
        x = ab.loc[ab['group'] == 'high_abs_contribution', 'Delta_AE']
        y = ab.loc[ab['group'] == 'low_abs_contribution', 'Delta_AE']
        diff, lo, hi = bootstrap_mean_diff_ci(x, y, B=3000 if REPORTING_MODE else 500, seed=seed)
        eff = pd.DataFrame([{
            'contrast': 'high_abs_contribution_minus_low_abs_contribution',
            'selection_rule': f'top {100*(1-high_q):.0f}% vs bottom {100*low_q:.0f}% by |ts-AIME temp|',
            'ablation_forecaster': 'manual S-Map-style local weighted linear endpoint model',
            'mean_difference_Delta_AE': diff,
            'bootstrap_CI_low': lo,
            'bootstrap_CI_high': hi,
            'cliffs_delta': cliffs_delta(x, y),
            'n_high': int(len(x)),
            'n_low': int(len(y)),
        }])
        save_csv(summary, 'E3_bike_ablation_temp_summary.csv')
        save_csv(eff, 'E3_bike_ablation_temp_effect_size.csv')
        save_table_tex(summary, 'tab_E3_bike_ablation_temp_summary.tex',
                       caption='Bike: one-step temperature ablation, grouped by high versus low absolute ts-AIME contribution.',
                       label='tab:E3_bike_ablation_summary')
        save_table_tex(eff, 'tab_E3_bike_ablation_temp_effect_size.tex',
                       caption='Bike: effect size for the high-minus-low temperature ablation contrast.',
                       label='tab:E3_bike_ablation_effect')
        plt.figure(figsize=(6.5, 4.6))
        data = [x.values, y.values]
        plt.boxplot(data, tick_labels=['high |ts-AIME temp|', 'low |ts-AIME temp|'])
        plt.ylabel('Δ absolute one-step forecast error after temp permutation')
        plt.title('Bike ablation: high versus low ts-AIME temperature contribution')
        save_fig('fig_E3_bike_ablation_temp_deltaAE_boxplot.png')
        plt.close()
    else:
        print('[warn] Bike ablation produced no valid endpoint windows; tables were not generated.')
    return ab

def run_bike_pipeline():
    dataset = 'E2_bike'
    df, raw = load_bike_sharing_day()
    features = ['temp', 'hum', 'windspeed']; target = 'cnt'
    save_dataset_tables(df, raw.rename(columns={'dteday': 'Date'}) if 'dteday' in raw.columns else raw,
                        dataset, [target] + features,
                        lags={'1day': 1, '7days': 7, '30days': 30, '1year': 365, '10years': 3650})
    eval_res = evaluate_forecast_horizons(df, dataset, target, features, HORIZONS_DAILY, HOLDOUT_FRAC_DAILY, maxE=10)
    plot_observed_vs_forecast(dataset, eval_res, tp=1, target_label='bike count')
    plot_metric_by_horizon(dataset, eval_res['metrics'], metric='RMSE')
    plot_metric_by_horizon(dataset, eval_res['metrics'], metric='rho')
    theta_mv = float(eval_res['params'].iloc[0]['theta_mv'])
    pred_full = full_series_smap_predictions(df, target, features, theta_mv=theta_mv, tp=1, include_target_in_columns=True)
    aligned = align_forecasts_with_source_features(df, pred_full, features, tp=1)
    tsa = rolling_tsaime_fast(aligned, features, window=W_BIKE, R=R_NULL_BIKE, period=7, step=TSAIME_STEP_DAILY, seed=SEED)
    save_tsaime_plot(dataset, 'full', tsa, features, window=W_BIKE, tp=1, null_label='7-day circular shift')
    save_tsaime_raw_outputs(dataset, 'full', tsa)
    significant_intervals_table(tsa, features, dataset, 'full', unit_label='n_days')
    significance_by_season_table(tsa, features, dataset, 'full')
    correlation_equivalence_check(tsa, features, dataset, 'full')
    # ts-AIME vs S-Map coefficient series from full multivariate S-Map.
    try:
        sm = ed.SMap(dataFrame=df[['Time'] + [target] + features], columns=' '.join([target] + features), target=target,
                     lib=f'1 {len(df)}', pred=f'1 {len(df)}', E=len([target] + features), embedded=True,
                     theta=theta_mv, Tp=1, showPlot=False)
        coef = sm.get('coefficients', pd.DataFrame()).copy()
        rows = []
        if len(coef):
            coef['Time'] = pd.to_numeric(coef['Time'], errors='coerce')
            G = tsa['global']
            for f in features:
                candidates = [c for c in coef.columns if c != 'Time' and str(c).strip().endswith(f)]
                if not candidates:
                    candidates = [c for c in coef.columns if c != 'Time' and f in str(c)]
                if candidates:
                    cc = coef[['Time', candidates[0]]].rename(columns={candidates[0]: 'coef'}).dropna()
                    mm = G[['Time', f]].merge(cc, on='Time', how='inner')
                    rows.append({'feature': f, 'corr_tsAIME_SMap_coef': safe_corr(mm[f], mm['coef']), 'coef_column': candidates[0],
                                 'interpretation': 'diagnostic contrast: rolling association vs local gradient'})
        comp = pd.DataFrame(rows)
        save_csv(comp, 'E2_bike_tsAIME_vs_SMap_coefficients.csv')
        save_table_tex(comp, 'tab_E2_bike_tsAIME_vs_SMap_coefficients.tex',
                       caption='Bike: diagnostic comparison between ts-AIME rolling associations and S-Map coefficient series.',
                       label='tab:E2_bike_aime_smap_coef')
    except Exception as e:
        print('[warn] S-Map coefficient comparison failed:', e)
    ab = ablation_temp_bike(df, tsa, eval_res, K=ABLATION_K, repeats=ABLATION_REPEATS, seed=SEED)
    return {'df': df, 'eval': eval_res, 'tsa': tsa, 'ablation': ab}

print('[ready] run_bike_pipeline()')


[ready] run_bike_pipeline()


## 10. Beijing PM2.5 experiment E6: descriptive statistics, horizons, ts-AIME, seasonal sign summary

**Hotfix note.** The revised run keeps `Is` and `Ir` in the descriptive statistics and optional forecasting covariate set, but excludes them from the main ts-AIME attribution table because they are extremely zero-inflated and can create undefined rolling correlations.

In [ ]:

def run_pm25_pipeline():
    dataset = 'E6_pm25'
    df, raw = load_beijing_pm25(max_years=PM25_MAX_YEARS, interpolate_limit_hours=PM25_INTERPOLATE_LIMIT_HOURS)
    features_model = ['TEMP', 'DEWP', 'Iws', 'Is', 'Ir']
    features_tsaime = ['TEMP', 'DEWP', 'Iws']  # Is/Ir are highly zero-inflated; keep them in descriptive tables but exclude from main attribution.
    target = 'PM2.5'
    save_dataset_tables(df, raw, dataset, [target] + features_model,
                        lags={'1hour': 1, '1day': 24, '7days': 24*7, '30days': 24*30,
                              '1year': 24*365, '10years': 24*3650})
    eval_res = evaluate_forecast_horizons(df, dataset, target, features_model, HORIZONS_HOURLY, HOLDOUT_FRAC_HOURLY, maxE=8)
    plot_observed_vs_forecast(dataset, eval_res, tp=1, target_label='PM2.5', max_points=24*60)
    plot_metric_by_horizon(dataset, eval_res['metrics'], metric='RMSE')
    plot_metric_by_horizon(dataset, eval_res['metrics'], metric='rho')
    theta_mv = float(eval_res['params'].iloc[0]['theta_mv'])
    pred_full = full_series_smap_predictions(df, target, features_model, theta_mv=theta_mv, tp=1, include_target_in_columns=True)
    aligned = align_forecasts_with_source_features(df, pred_full, features_tsaime, tp=1)
    tsa = rolling_tsaime_fast(aligned, features_tsaime, window=W_PM25, R=R_NULL_PM25, period=24*7,
                              step=TSAIME_STEP_PM25, seed=SEED)
    save_tsaime_plot(dataset, 'full', tsa, features_tsaime, window=W_PM25, tp=1, null_label='168-hour circular shift')
    save_tsaime_raw_outputs(dataset, 'full', tsa)
    significant_intervals_table(tsa, features_tsaime, dataset, 'full', unit_label='n_hours_or_steps')
    significance_by_season_table(tsa, features_tsaime, dataset, 'full')
    correlation_equivalence_check(tsa, features_tsaime, dataset, 'full')
    return {'df': df, 'raw': raw, 'eval': eval_res, 'tsa': tsa}

print('[ready] run_pm25_pipeline()')


[ready] run_pm25_pipeline()


## 11. Run all experiments

In [ ]:

RUN_SYNTH = True
RUN_BIKE = True
RUN_PM25 = True

all_results = {}
if RUN_SYNTH:
    print('\n========== Running E1 synthetic ==========' )
    all_results['E1_synth'] = run_synthetic_pipeline()
if RUN_BIKE:
    print('\n========== Running E2/E3 Bike ==========' )
    all_results['E2_bike'] = run_bike_pipeline()
if RUN_PM25:
    print('\n========== Running E6 PM2.5 ==========' )
    all_results['E6_pm25'] = run_pm25_pipeline()

print('\nAll selected experiments complete.')
print('Figures:', FIG_DIR)
print('TeX tables:', TEX_DIR)
print('CSV tables:', CSV_DIR)



========== Running E1 synthetic ==========
[saved csv] /content/drive/MyDrive/Colab Notebooks/Research/EDM/results_v7_publication_ready_operator_validation/outputs/csv/E1_synth_data_overview.csv
[saved csv] /content/drive/MyDrive/Colab Notebooks/Research/EDM/results_v7_publication_ready_operator_validation/outputs/csv/E1_synth_primary_statistics.csv
[saved csv] /content/drive/MyDrive/Colab Notebooks/Research/EDM/results_v7_publication_ready_operator_validation/outputs/csv/E1_synth_seasonal_statistics.csv
[saved csv] /content/drive/MyDrive/Colab Notebooks/Research/EDM/results_v7_publication_ready_operator_validation/outputs/csv/E1_synth_autocorrelation.csv
[saved csv] /content/drive/MyDrive/Colab Notebooks/Research/EDM/results_v7_publication_ready_operator_validation/outputs/csv/E1_synth_cross_correlation.csv
[saved tex] /content/drive/MyDrive/Colab Notebooks/Research/EDM/results_v7_publication_ready_operator_validation/outputs/tex/tab_E1_synth_data_overview.tex
[saved tex] /content/dr

## 12. Artifact manifest and zip outputs

In [ ]:

def build_manifest():
    rows = []
    for kind, folder in [('figure', FIG_DIR), ('tex', TEX_DIR), ('csv', CSV_DIR), ('log', LOG_DIR)]:
        for p in sorted(folder.glob('*')):
            if p.is_file():
                rows.append({'kind': kind, 'file': p.name, 'path': str(p), 'bytes': p.stat().st_size})
    manifest = pd.DataFrame(rows)
    save_csv(manifest, 'artifact_manifest.csv')
    if len(manifest):
        save_table_tex(manifest[['kind', 'file', 'bytes']], 'tab_artifact_manifest.tex',
                       caption='Generated artifacts from the revision notebook.',
                       label='tab:artifact_manifest', max_rows=80)
    return manifest


def publication_readiness_check():
    """Check that the generated artifacts are manuscript-safe.

    The check intentionally prevents common failure modes observed in earlier runs:
    stale PM2.5 Is/Ir attribution files, weak synthetic recovery, and literal NaN in TeX.
    """
    problems = []
    notes = []
    if NOTEBOOK_VERSION != 'v7_publication_ready_operator_validation':
        problems.append(f'Wrong notebook version running: {NOTEBOOK_VERSION}. Restart runtime and open the v7 notebook.')
    if 'results_v7_publication_ready_operator_validation' not in str(OUTPUT_DIR):
        problems.append(f'Wrong output directory: {OUTPUT_DIR}. This indicates stale cells from an older notebook.')
    rec = CSV_DIR / 'E1_synth_recovery_correlations.csv'
    if rec.exists():
        _df = pd.read_csv(rec)
        print('[check] E1 recovery correlations:')
        print(_df.to_string(index=False))
        vals = dict(zip(_df['feature'], _df['corr_with_truth']))
        if vals.get('x1', 0) < 0.60 or vals.get('x2', 0) < 0.60:
            problems.append(f"Synthetic recovery below publication threshold in {NOTEBOOK_VERSION}: {vals}. Expected both x1 and x2 >= 0.60.")
    else:
        problems.append('Missing E1_synth_recovery_correlations.csv')

    stale_pm25 = sorted(CSV_DIR.glob('E6_pm25_full_significant_intervals_I*.csv'))
    stale_pm25 = [p for p in stale_pm25 if p.name in ['E6_pm25_full_significant_intervals_Is.csv', 'E6_pm25_full_significant_intervals_Ir.csv']]
    if stale_pm25:
        problems.append('Stale PM2.5 Is/Ir attribution files exist: ' + ', '.join(p.name for p in stale_pm25))
    sig = CSV_DIR / 'E6_pm25_full_significance_by_season.csv'
    if sig.exists():
        s = pd.read_csv(sig)
        features_seen = sorted(s['feature'].dropna().unique().tolist())
        print('[check] PM2.5 attribution features:', features_seen)
        if any(f in set(features_seen) for f in ['Is', 'Ir']):
            problems.append('PM2.5 significance table still contains Is/Ir; this indicates the old notebook or stale outputs.')
    else:
        problems.append('Missing E6 PM2.5 significance table')

    # TeX safety: no literal NaN/inf should appear in publication tables.
    bad_tex = []
    for p in TEX_DIR.glob('*.tex'):
        txt = p.read_text(encoding='utf-8', errors='ignore')
        if any(tok in txt for tok in [' NaN', 'NaN ', ' nan', 'nan ', ' inf', 'inf ']):
            bad_tex.append(p.name)
    if bad_tex:
        problems.append('Literal NaN/inf found in TeX files: ' + ', '.join(bad_tex[:10]))

    # Ablation is a sensitivity analysis; report its status rather than forcing significance.
    eff = CSV_DIR / 'E3_bike_ablation_temp_effect_size.csv'
    if eff.exists():
        e = pd.read_csv(eff)
        print('[check] Bike ablation effect size:')
        print(e.to_string(index=False))
        if len(e):
            lo = float(e['bootstrap_CI_low'].iloc[0])
            hi = float(e['bootstrap_CI_high'].iloc[0])
            if lo > 0:
                notes.append('Bike high-vs-low ablation supports a main-text predictive-sensitivity claim.')
            else:
                notes.append('Bike high-vs-low ablation is valid but CI overlaps zero; use as supplementary sensitivity analysis.')
    else:
        notes.append('Bike ablation table not found.')

    readiness = pd.DataFrame([{
        'notebook_version': NOTEBOOK_VERSION,
        'run_id': RUN_ID,
        'status': 'PASS' if not problems else 'FAIL',
        'problems': '; '.join(problems) if problems else '',
        'notes': '; '.join(notes),
    }])
    save_csv(readiness, 'publication_readiness_report.csv')
    save_table_tex(readiness, 'tab_publication_readiness_report.tex',
                   caption='Automatic publication-readiness checks for generated artifacts.',
                   label='tab:publication_readiness')
    if problems:
        raise RuntimeError('\n'.join(problems))
    print('[check] publication-readiness guard passed.')

publication_readiness_check()
manifest = build_manifest()
print(manifest.head(20))

zip_path = OUTPUT_DIR / f'revision_outputs_{NOTEBOOK_VERSION}_{RUN_ID}.zip'
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for folder in [FIG_DIR, TEX_DIR, CSV_DIR, LOG_DIR]:
        for p in folder.glob('*'):
            if p.is_file():
                zf.write(p, arcname=str(p.relative_to(OUTPUT_DIR)))
print('[saved zip]', zip_path)

# In Colab, uncomment to download:
# from google.colab import files
# files.download(str(zip_path))


[check] E1 recovery correlations:
feature ground_truth  corr_with_truth
     x1   true_alpha         0.756107
     x2    true_beta         0.740168
[check] PM2.5 attribution features: ['DEWP', 'Iws', 'TEMP']
[check] Bike ablation effect size:
                                        contrast                          selection_rule                                     ablation_forecaster  mean_difference_Delta_AE  bootstrap_CI_low  bootstrap_CI_high  cliffs_delta  n_high  n_low
high_abs_contribution_minus_low_abs_contribution top 15% vs bottom 15% by |ts-AIME temp| manual S-Map-style local weighted linear endpoint model                  46.73558        -11.975533         106.712468      0.162222      60     60
[saved csv] /content/drive/MyDrive/Colab Notebooks/Research/EDM/results_v7_publication_ready_operator_validation/outputs/csv/publication_readiness_report.csv
[saved tex] /content/drive/MyDrive/Colab Notebooks/Research/EDM/results_v7_publication_ready_operator_validation/outputs/tex/


## 13. Notes for manuscript revision

Recommended wording based on the reviewer comments:

- Replace **"attach statistically grounded attributions"** with **"compute rolling, forecast-aligned attribution scores"**.
- State explicitly that, in the scalar-output case under in-window z-standardization, \(A_t^\dagger\) is a **rolling correlation-like score** between the EDM forecast sequence and each covariate.
- Use **association / co-variation / forecast-aligned contribution** rather than causal terms such as influence or effect.
- For FDR, write: **"Benjamini--Hochberg was applied across covariates at each time point; contiguous significant intervals are descriptive summaries of pointwise discoveries."**
- In the forecast section, report **B1 mean predictor** and **B2 persistence predictor** for each horizon.
